# Phase 2 – Step 2.8: Risk Analyst Agent — Demo

**Purpose:** watch the Chain-of-Thought agent work on real cases and judge its output yourself.

**What you will see**
1. The exact (masked, PII-free) message the model receives
2. The agent on one case per planted pattern, plus a clean customer and a no-activity customer
3. Its answers next to the *hidden* ID-prefix labels it never saw
4. One full reasoning chain, and the audit trail it left
5. Tokens, latency and an estimated cost

> ⚠️ Section 3 makes **6 real API calls** (about 25k tokens, well under one cent on `gpt-4o-mini`).

## 0. Setup

In [ ]:
# Phase 2 – Step 2.8: imports, data, and one agent wired to the real client
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import textwrap

import pandas as pd

from src import config
from src.case_formatter import format_case_for_llm
from src.foundation_sar import DataLoader, ExplainabilityLogger
from src.llm_client import get_llm_client
from src.risk_analyst_agent import RiskAnalysisError, RiskAnalystAgent

AUDIT_FILE = config.LOG_DIR / "demo_risk_analyst.jsonl"   # logs/ is git-ignored
logger = ExplainabilityLogger(AUDIT_FILE)
cases = {c.customer.customer_id: c for c in DataLoader(logger).build_all_cases()}
agent = RiskAnalystAgent(get_llm_client(), logger)
print(f"{len(cases)} cases | model={agent.model} | prompt={agent.prompt_version} | "
      f"temperature={agent.temperature} | seed={agent.seed}")

## 1. Choose demo cases
The transaction-ID prefixes are the **answer key** planted in the synthetic data. We use them here
only to *pick* and later *grade* cases. The model never sees them (Step 2.3 masking).

In [ ]:
# Phase 2 – Step 2.8: hidden label per customer = the planted prefix(es) on their transactions
def hidden_label(case):
    prefixes = {t.transaction_id.split("_")[0] for t in case.transactions} - {"TXN"}
    if not case.transactions:
        return "(no activity)"
    return ", ".join(sorted(prefixes)) or "(none planted)"


demo_ids = []
for label in ["STRUCT", "LARGE", "RAPID", "GEO", "FREQ"]:
    demo_ids.append(next(cid for cid, c in cases.items() if hidden_label(c) == label))
demo_ids.append("CUST_0002")                                                  # clean customer
demo_ids.append(next(cid for cid, c in cases.items() if not c.transactions))  # no activity
pd.DataFrame([{"customer": cid, "hidden_label": hidden_label(cases[cid]),
               "transactions": len(cases[cid].transactions)} for cid in demo_ids])

## 2. What the model actually receives
Notice: `T001…` instead of real IDs, **age** instead of date of birth, no SSN, address or phone,
and a *neutral* activity summary with no red flags pointed out (Stage 02 decision 1).

In [ ]:
# Phase 2 – Step 2.8: the masked user message for the structuring customer
formatted = format_case_for_llm(cases[demo_ids[0]])
lines = formatted.text.splitlines()
print("\n".join(lines[:24]))
print("...")
print("\n".join(l for l in lines if l.startswith("T") and "Cash_Deposit" in l))
print(f"\n~{len(formatted.text) // 4} tokens | {len(formatted.id_map)} masked refs, "
      f"e.g. T001 -> {formatted.id_map.get('T001')}")

## 3. Run the agent  ⚠️ real API calls

In [ ]:
# Phase 2 – Step 2.8: analyse each demo case; failures are caught, as the workflow will do
rows, results = [], {}
for cid in demo_ids:
    case = cases[cid]
    try:
        result = agent.analyze_case(case)
    except RiskAnalysisError as exc:
        rows.append({"customer": cid, "hidden_label": hidden_label(case), "error": exc.reason})
        continue
    run = agent.last_run
    results[cid] = result
    rows.append({
        "customer": cid,
        "hidden_label": hidden_label(case),
        "classification": result.classification,
        "risk_level": result.risk_level,
        "confidence": result.confidence_score,
        "method": run.method,
        "tokens": run.token_usage.get("total_tokens", 0),
        "latency_s": round(run.latency_ms / 1000, 1),
        "attempts": run.attempts,
    })
summary = pd.DataFrame(rows)
summary

**How to read this table:** compare `classification` with `hidden_label`. Expected mappings:
`STRUCT` → Structuring; `LARGE`/`RAPID` → Money_Laundering is plausible; `GEO`/`FREQ` are open to
interpretation (Structuring, Fraud or Other can all be argued). The clean customer should be Low, and
the no-activity customer is `rule_based` with 0 tokens. Disagreements are *findings*, not test failures:
Stage 08 turns this comparison into a proper evaluation.

## 4. One full reasoning chain

In [ ]:
# Phase 2 – Step 2.8: read the 5-step Chain-of-Thought for the structuring case
r = results.get(demo_ids[0])
if r:
    print(f"{r.classification} | {r.risk_level} | confidence {r.confidence_score}\n")
    for step in r.reasoning.split("Step ")[1:]:
        print(textwrap.fill("Step " + step.strip(), width=110), "\n")
    print("Key indicators:")
    for k in r.key_indicators:
        print(" •", textwrap.fill(k, width=106, subsequent_indent="   "))

**Check the facts yourself:** compare every amount and date in the indicators with the transaction
lines in section 2. In Step 2.4 the model once attached the wrong date to an amount. *Valid JSON
is not the same as correct facts.* Catching that automatically is a Stage 08 job.

## 5. The audit trail

In [ ]:
# Phase 2 – Step 2.8: the RiskAnalyst events this notebook just wrote
events = [json.loads(l) for l in AUDIT_FILE.read_text(encoding="utf-8").splitlines()]
recent = [e for e in events if e["agent_type"] == "RiskAnalyst"][-len(demo_ids):]
pd.DataFrame([{k: e.get(k) for k in ["case_id", "success", "method", "model", "seed",
                                     "system_fingerprint", "attempts", "prompt_version"]}
              | {"total_tokens": (e.get("token_usage") or {}).get("total_tokens")}
              for e in recent])

## 6. Tokens, latency and cost
Prices below are **assumptions** for `gpt-4o-mini` (USD per 1M tokens). Check the current price list
before relying on them. The audit log has exact token counts, so cost can always be recomputed.

In [ ]:
# Phase 2 – Step 2.8: estimate cost from the audit log's token counts
PRICE_PER_M = {"prompt": 0.15, "completion": 0.60}   # assumed USD / 1M tokens — verify
usage = [e.get("token_usage") or {} for e in recent]
prompt_t = sum(u.get("prompt_tokens", 0) for u in usage)
completion_t = sum(u.get("completion_tokens", 0) for u in usage)
cost = prompt_t / 1e6 * PRICE_PER_M["prompt"] + completion_t / 1e6 * PRICE_PER_M["completion"]
llm_runs = summary[summary.get("method", pd.Series(dtype=str)) == "llm"]
print(f"prompt tokens {prompt_t:,} | completion tokens {completion_t:,} | est. cost ${cost:.4f}")
if len(llm_runs):
    print(f"avg latency {llm_runs.latency_s.mean():.1f}s per LLM case -> "
          f"~{llm_runs.latency_s.mean() * 119 / 60:.0f} min for all 119 active cases, one at a time")

## 7. Takeaways
- **Masking works:** the model reasons from amounts, dates and patterns, not from planted labels.
- **Chain-of-Thought leaves an audit trail:** every conclusion is tied to Step 1–5 reasoning.
- **Validated ≠ correct:** check the facts in the indicators. Faithfulness grading comes in Stage 08.
- **Reproducibility needs work:** `temperature=0` alone wasn't enough; `seed` + `system_fingerprint` help.
- **Cost and latency are small per case but add up:** 119 active cases take minutes sequentially,
  which is why the two-stage design (Stage 04) only sends approved cases to the next agent.